# Phase 0/1 — CAMEL(S) Rating: Dataset Research & Feature Engineering
### Thin walkthrough notebook — the real logic lives in `src/camel_sentinel/`

**What this notebook does**: calls the `camel_sentinel` package to pull (or synthesize) FDIC bank data, build CAMEL ratios, score a composite proxy rating, and run first-pass EDA on the result — then shows the numbers and plots.

**Why the logic isn't written inline here**: so the same tested functions are reusable from the FastAPI backend and the agent (built in later phases) instead of being copy-pasted between them. This notebook is meant to be read top to bottom as a *walkthrough* of Phase 1, not as the implementation itself.

**Where to look for more context**:
- [`../docs/00_problem_statement.md`](../docs/00_problem_statement.md) — the problem this whole project solves, and why a proxy label is necessary
- [`../docs/01_glossary.md`](../docs/01_glossary.md) — every term used below (CAMEL components, composite z-score, proxy rating, ...)
- [`../docs/02_data_sources.md`](../docs/02_data_sources.md) — what the FDIC API gives us and why, plus the network-access caveat referenced below
- [`../src/camel_sentinel/data/fdic_client.py`](../src/camel_sentinel/data/fdic_client.py) — the data-pull + synthetic-fallback code called in Section 1-2
- [`../src/camel_sentinel/features/camel_ratios.py`](../src/camel_sentinel/features/camel_ratios.py) — the ratio + composite-scoring code called in Section 3-4

> **Network note**: `fdic_client.try_live_pull()` below calls the live FDIC API and reports whether it succeeded. If your environment can't reach `banks.data.fdic.gov` (some sandboxes/CI runners block it), it falls back to `fdic_client.build_synthetic_sample()` automatically — same schema, realistic distributions, clearly labeled — so every cell below still runs either way. Don't treat the synthetic sample as a finding about real banks.


In [1]:
# --- Setup ---
import sys
import warnings
from pathlib import Path

import pandas as pd

# Make the src/ package importable from notebooks/ without installing it —
# sys.path.insert(0, ...) puts "../src" at the front of Python's module
# search path, so `import camel_sentinel...` below finds the package we
# just built instead of failing with ModuleNotFoundError.
SRC_PATH = str(Path("..").resolve() / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from camel_sentinel.data import fdic_client
from camel_sentinel.features import camel_ratios

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("Environment ready — camel_sentinel package imported from", SRC_PATH)


Environment ready — camel_sentinel package imported from C:\Users\raush\Downloads\AI-ML-Project-Afgan\Project\src


## 1. Pull real data from the FDIC BankFind Suite API

Three endpoints (see [`docs/02_data_sources.md`](../docs/02_data_sources.md) for what each gives us and why), called through `fdic_client.try_live_pull()`:

- `institutions` — bank identity/metadata
- `financials` — quarterly Call Report fields (this is where the CAMEL ratio inputs live)
- `failures` — historical bank failures (our proxy-label source)

`try_live_pull()` returns `live_pull_ok=False` instead of raising if the API can't be reached, so the notebook can decide whether to fall back to the synthetic sample in Section 2.


In [2]:
# fdic_client.try_live_pull(report_date=...) — see its docstring in
# src/camel_sentinel/data/fdic_client.py for what each parameter means and
# exactly which fields get pulled from each endpoint.
institutions, financials, failures, LIVE_PULL_OK = fdic_client.try_live_pull(
    report_date="20231231"  # Q4 2023 Call Report snapshot — see docstring for the format
)

print("Live pull succeeded:", LIVE_PULL_OK)


Live pull succeeded: True


## 2. Fallback: synthetic sample with the same schema

Used **only** if the live pull above didn't succeed. `fdic_client.build_synthetic_sample()` generates data with the *same fields and realistic distributions* as the live FDIC data, so every downstream cell — ratio construction, composite scoring, EDA, plots — runs and shows real output now. **Do not treat this synthetic sample as a finding about real banks** (see [`docs/02_data_sources.md`](../docs/02_data_sources.md)) — swap in the live pull the moment you run this notebook somewhere with internet access.


In [3]:
if not LIVE_PULL_OK:
    # fdic_client.build_synthetic_sample(n_banks=..., seed=...) — see its
    # docstring for exactly how each ratio is distributed and how the
    # synthetic failure flag is derived.
    institutions, financials, failures = fdic_client.build_synthetic_sample(
        n_banks=1500,  # optional — how many synthetic banks to generate
        seed=42,       # optional — fixed so this notebook's output is reproducible
    )
    print(
        f"Synthetic sample built: {len(institutions)} banks, {len(failures)} flagged as failed "
        f"({len(failures) / len(institutions):.1%}) — matches real-world FDIC failure-rate order of magnitude."
    )


## 3. Merge & build the CAMEL(S) ratio table

`camel_ratios.build_camel_ratios()` follows the ratio recipe documented in [`docs/01_glossary.md`](../docs/01_glossary.md) and `docs/CAMEL-Rating-ML-Project-Blueprint.md` Section 3, and in the inline comments of [`src/camel_sentinel/features/camel_ratios.py`](../src/camel_sentinel/features/camel_ratios.py) — including the explicit caveat on the Management ("M") component, which has no clean financial-statement proxy.


In [4]:
# camel_ratios.build_camel_ratios(institutions, financials, failures) — see
# its docstring for the required shape of each input DataFrame and exactly
# which raw Call Report field feeds which CAMEL ratio.
camel = camel_ratios.build_camel_ratios(institutions, financials, failures)
camel.head()


,CERT,NAME,STALP,ASSET,tier1_ratio,npl_ratio,provision_ratio,efficiency_ratio,roa,roe,nim,loan_deposit_ratio,liquid_asset_ratio,loan_concentration,failed
0,10004,Ergo Bank,WI,240478,9.76,0.530194,1457.0,66.728579,0.893430,9.84,3.727746,99.118581,NaN,79.215562,0
1,10011,Woodford State Bank,WI,442468,10.67,0.913060,1918.0,74.865952,0.748840,15.54,3.134537,88.020311,NaN,68.049893,0
2,10012,The Portage County Bank,WI,222026,NaN,0.515255,1151.0,77.011853,0.618052,6.18,2.909937,59.149344,NaN,49.983335,0
3,10015,Security Bank,WI,181780,NaN,0.652437,1778.0,58.105355,1.232162,9.95,3.827291,101.450559,NaN,85.067114,0
4,10044,National Exchange Bank and Trust,WI,2786163,NaN,1.620221,52166.0,47.602329,1.481281,9.58,3.712141,74.429198,NaN,58.878680,0


## 4. Rule-based composite CAMEL score (0–100, higher = healthier)

`camel_ratios.score_composite()` z-scores each component in the "healthier direction" (see "Composite z-score" in [`docs/01_glossary.md`](../docs/01_glossary.md)), averages them, and rescales — this becomes the **regression target**; bucketed into quintiles it becomes the **1–5 rating classification target**. This is the literature-standard approach precisely because no ground-truth rating exists to train against directly (see [`docs/00_problem_statement.md`](../docs/00_problem_statement.md)).


In [5]:
# camel_ratios.score_composite(camel) — see its docstring for exactly how
# composite_z / composite_score_0_100 / proxy_rating_1_5 are derived.
camel = camel_ratios.score_composite(camel)

camel[["NAME", "composite_score_0_100", "proxy_rating_1_5", "failed"]].sort_values(
    "composite_score_0_100"
).head(10)


,NAME,composite_score_0_100,proxy_rating_1_5,failed
4070,NaN,0.0,5,0
4073,NaN,5.0,5,0
4067,NaN,7.8,5,0
4248,NaN,9.5,5,0
4074,NaN,9.9,5,0
3959,NaN,11.9,5,0
3264,NaN,12.2,5,0
2353,NaN,12.9,5,1
3877,NaN,13.4,5,0
4059,NaN,13.8,5,0


## 5. Sanity check: does the composite proxy actually track real distress?

Cross-tab the rule-based rating against the (synthetic here, real once live) failure flag. If ratings 4–5 capture a disproportionate share of failures, the composite score is doing its job as an early-warning proxy — this is the single most important validation step in the whole pipeline, and the one to show the client first.


In [6]:
crosstab = pd.crosstab(camel["proxy_rating_1_5"], camel["failed"], normalize="index") * 100
crosstab.columns = ["pct_not_failed", "pct_failed"]
print(crosstab.round(1))
print()
print("Failure rate should climb monotonically from rating 1 -> rating 5 if the composite score is well-formed.")


                  pct_not_failed  pct_failed
proxy_rating_1_5                            
1                           99.8         0.2
2                           99.7         0.3
3                           99.8         0.2
4                           99.8         0.2
5                           99.4         0.6

Failure rate should climb monotonically from rating 1 -> rating 5 if the composite score is well-formed.


> **A real join bug this fixed, worth knowing about**: an earlier version of this pipeline joined `institutions` (pulled with an `ACTIVE:1` filter) as the *base* table onto `financials`. On live data that silently dropped almost every bank that had ever failed — a failed bank is, by definition, not in today's active roster, so it never survived that join, and the crosstab above showed a near-zero failure rate in every rating bucket regardless of rating quality. `camel_ratios.build_camel_ratios()` now treats `financials` (every bank that filed a Call Report that quarter) as the base and left-joins institution metadata onto it instead — see the "Notes" section of its docstring. `fdic_client.fetch_institutions()` also no longer filters to `ACTIVE:1` by default, for the same reason.
>
> One caveat remains, by design, for a single-quarter pull: only a few dozen of the ~4,700 banks in one quarter's snapshot show up anywhere in the full FDIC failure history, so the crosstab above is noisy — a rating bucket can look artificially high or low purely from having 2-3 failures instead of 3-4. That's exactly why the blueprint's Phase 2 calls for pulling **multiple quarters** and building a proper "failed within N quarters of this report" label instead of relying on one snapshot.


## 6. First-pass EDA

Distribution of each CAMEL component, and how the composite score separates failed vs. non-failed institutions.


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
comp_display = {
    "tier1_ratio": "Tier 1 Capital Ratio (%)",
    "npl_ratio": "NPL Ratio (%)",
    "efficiency_ratio": "Efficiency Ratio (%)",
    "roa": "ROA (%)",
    "loan_deposit_ratio": "Loan / Deposit Ratio (%)",
    "loan_concentration": "Loan Concentration (%)",
}
for ax, (col, label) in zip(axes.ravel(), comp_display.items()):
    finite_vals = camel[col].replace([float("inf"), float("-inf")], pd.NA).dropna()
    ax.hist(finite_vals, bins=30, color="#2b6cb0", alpha=0.85)
    ax.set_title(label, fontsize=10)
    ax.set_ylabel("bank count")
plt.tight_layout()
plt.savefig("camel_component_distributions.png", dpi=110)
plt.show()
print("Saved: camel_component_distributions.png")


Saved: camel_component_distributions.png


In [8]:
fig, ax = plt.subplots(figsize=(8, 5))
for flag, label, color in [(0, "Not failed", "#2b6cb0"), (1, "Failed", "#c53030")]:
    subset = camel.loc[camel["failed"] == flag, "composite_score_0_100"]
    ax.hist(subset, bins=25, alpha=0.6, label=f"{label} (n={len(subset)})", color=color, density=True)
ax.set_xlabel("Composite CAMEL score (0-100, higher = healthier)")
ax.set_ylabel("density")
ax.set_title("Composite score separation: failed vs. not-failed institutions")
ax.legend()
plt.tight_layout()
plt.savefig("camel_composite_separation.png", dpi=110)
plt.show()
print("Saved: camel_composite_separation.png")


Saved: camel_composite_separation.png


## 7. What's next

This notebook completes the **Phase 0/1 foundation**: data acquisition + feature engineering, now living in `src/camel_sentinel/` and reused (not copy-pasted) from here on. See [`docs/03_module_mapping.md`](../docs/03_module_mapping.md) for how each remaining phase maps to the curriculum, and `docs/CAMEL-Rating-ML-Project-Blueprint.md` Section 5 for the full phase table.

1. Swap the synthetic fallback for the live FDIC pull (Section 1) — run this notebook on a machine with internet access, ideally with a free FDIC API key for higher rate limits.
2. Pull **multiple quarters** (not just one snapshot) so the failure label can be built properly as "failed within N quarters of this report" rather than a same-quarter flag.
3. Cross-validate the composite-score recipe against the Kaggle sandbox datasets (Polish bankruptcy, corporate credit rating) referenced in [`docs/02_data_sources.md`](../docs/02_data_sources.md), to check the ratio→rating relationship generalizes beyond US banks.
4. Move to Phase 3: train the actual regression/classification/clustering/anomaly-detection models on top of this feature table (not just the rule-based composite) in a new `notebooks/01_eda.ipynb` / `02_feature_engineering.ipynb` / `03_baseline_modeling.ipynb` sequence, then Phase 5 XAI (SHAP/LIME/counterfactuals) on top of *that*.
5. Keep the "Management" caveat visible everywhere downstream — it's a proxy (efficiency ratio), not the real supervisory judgment component, and should be called out in every explanation the agent generates.
